<h1 style="text-align: center; font-weight: bold;">TRAINING XGBOOST MODEL</h1>

## Loading Dataset

In [1]:
# Import libraries
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import KFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import classification_report, confusion_matrix
from tabulate import tabulate

In [2]:
# Load proccessed dataset
data = pd.read_csv("../data/processed_train.csv")
# Load test dataset
test_data = pd.read_csv("../data/processed_test.csv")

# Drop id column
data = data.drop(columns=['id'])
# Drop Name and City columns as they are not useful for modeling
data.drop(columns=['Name', 'City'], inplace=True)
test_data.drop(columns=['Name', 'City'], inplace=True)

# Keep test id column separately
test_ids = test_data['id']
# Drop test id column
test_data = test_data.drop(columns=['id'])

data.head()

,Gender,Age,Working Professional or Student,Profession,Academic Pressure,Work Pressure,CGPA,Study Satisfaction,Job Satisfaction,Sleep Duration,Dietary Habits,Degree,Have you ever had suicidal thoughts ?,Work/Study Hours,Financial Stress,Family History of Mental Illness,Overall Stress Level,Depression
0,0,49.0,1,Chef,0.0,5.0,0.0,0.0,2.0,9.0,Healthy,BHM,0,1.0,2.0,0,5.0,0
1,1,26.0,1,Teacher,0.0,4.0,0.0,0.0,3.0,4.0,Unhealthy,LLB,1,7.0,3.0,0,4.0,1
2,1,33.0,0,Not Applicable,5.0,0.0,5.5,2.0,0.0,5.5,Healthy,B.Pharm,1,3.0,1.0,0,4.0,1
3,1,22.0,1,Teacher,0.0,5.0,0.0,0.0,1.0,4.0,Moderate,BBA,1,10.0,1.0,1,5.0,1
4,0,30.0,1,Business Analyst,0.0,1.0,0.0,0.0,1.0,5.5,Unhealthy,BBA,1,9.0,4.0,1,4.0,0


## Define Training Parameters Constants

In [3]:
LABEL_COL = 'Depression'
NUMERIC_COLS = ['Age', 'Academic Pressure', 'Work Pressure', 'CGPA', 'Study Satisfaction', 'Job Satisfaction',
                'Sleep Duration', 'Work/Study Hours', 'Financial Stress', 'Overall Stress Level']
CATEGORICAL_COLS = ['Profession', 'Dietary Habits', 'Degree']

NUM_FOLD = 10
LR = 1e-3
L2_REG = 1e-4
MAX_DEPTH = 20
NUM_EPOCHS = 500
PATIENCE = 20
MIN_CHILD_WEIGHT = 5
COSAMPLE_RATE = 0.8
DATA_RATE = 0.8

## Data Preprocessing

In [4]:
# Separate features and labels
X = data.drop(columns=[LABEL_COL])
y = data[LABEL_COL]

all_data = pd.concat([X, test_data], axis=0) # For consistent encoding

# Normalize numeric features
scaler = StandardScaler()
all_data[NUMERIC_COLS] = scaler.fit_transform(all_data[NUMERIC_COLS])
X[NUMERIC_COLS] = all_data.iloc[:len(X)][NUMERIC_COLS]
test_data[NUMERIC_COLS] = all_data.iloc[len(X):][NUMERIC_COLS]

# Encode categorical features
for col in CATEGORICAL_COLS:
    encoder = LabelEncoder()
    all_data[col] = encoder.fit_transform(all_data[col])
    X[col] = all_data.iloc[:len(X)][col]
    test_data[col] = all_data.iloc[len(X):][col]
    
# Calculate class weights to handle class imbalance
scale_pos_weight = (y == 0).sum() / (y == 1).sum()
print(f"\nScale pos weight: {scale_pos_weight:.4f}")


Scale pos weight: 4.5032


## K-Fold Cross Validation Training

In [5]:
# K-fold Cross Validation Training
kf = KFold(n_splits=NUM_FOLD, shuffle=True, random_state=42)

fold = 1 # Fold counter
classification_reports = [] # To store classification reports for each fold
best_iterations = [] # To store best iterations for each fold

for train_index, val_index in kf.split(X):
    print(f"Training fold {fold}...")
    X_train, X_val = X.iloc[train_index], X.iloc[val_index]
    y_train, y_val = y.iloc[train_index], y.iloc[val_index]

    # Define XGB classifier model with parameters
    dtrain = xgb.XGBClassifier(
        learning_rate=LR,
        max_depth=MAX_DEPTH,
        n_estimators=NUM_EPOCHS,
        reg_lambda=L2_REG,  # L2 regularization
        eval_metric='error',  # Classification error rate (1 - accuracy)
        early_stopping_rounds=PATIENCE,
        enable_categorical=False,  # Categorical features are already encoded
        scale_pos_weight=scale_pos_weight,  # Handle class imbalance
        random_state=42,
        min_child_weight = MIN_CHILD_WEIGHT, # Minimum sum of instance weight needed in a child
        colsample_bytree = COSAMPLE_RATE, # Subsample ratio of columns when constructing each tree
        subsample = DATA_RATE # Subsample ratio of the training instances
    )
    
    # Train with early stopping using the validation set
    dtrain.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        verbose=False
    )

    # Store best iteration
    best_iterations.append(dtrain.get_booster().best_iteration)
    print(f"Best iteration for fold {fold}: {dtrain.get_booster().best_iteration}")
    # Evaluate on validation set
    y_pred = dtrain.predict(X_val)
    cr = classification_report(y_val, y_pred, output_dict=True, zero_division=0.0)
    cm = confusion_matrix(y_val, y_pred)
    print(f"Fold {fold} results:")
    print(cr)
    print("Confusion Matrix:")
    print(cm)
    
    # Store report
    classification_reports.append(cr)

    print("\n")
    fold += 1

Training fold 1...
Best iteration for fold 1: 58
Fold 1 results:
{'0': {'precision': 0.9786066922654965, 'recall': 0.9298123697011814, 'f1-score': 0.9535857461024498, 'support': 11512.0}, '1': {'precision': 0.7420178799489144, 'recall': 0.90852228303362, 'f1-score': 0.8168717047451669, 'support': 2558.0}, 'accuracy': 0.9259417199715707, 'macro avg': {'precision': 0.8603122861072054, 'recall': 0.9191673263674007, 'f1-score': 0.8852287254238084, 'support': 14070.0}, 'weighted avg': {'precision': 0.9355936018670732, 'recall': 0.9259417199715707, 'f1-score': 0.9287304143475152, 'support': 14070.0}}
Confusion Matrix:
[[10704   808]
 [  234  2324]]


Training fold 2...
Best iteration for fold 2: 2
Fold 2 results:
{'0': {'precision': 0.9621911544887013, 'recall': 0.9537214572075998, 'f1-score': 0.9579375848032564, 'support': 11474.0}, '1': {'precision': 0.8031145717463849, 'recall': 0.8343605546995377, 'f1-score': 0.8184394483279803, 'support': 2596.0}, 'accuracy': 0.9316986496090973, 'macro 

## Cross Validation Results

In [6]:
avg_report = {}

# Extract average metrics across folds
for key in classification_reports[0].keys():
    if key in ['0', '1', 'macro avg', 'weighted avg']: # Average the metric sections
        avg_report[key] = {}
        for metric in classification_reports[0][key].keys(): # Iterate through metrics ('precision', 'recall', etc.)
            avg_report[key][metric] = np.mean([report[key][metric] for report in classification_reports])
    elif key == 'accuracy':
        avg_report[key] = np.mean([report[key] for report in classification_reports])

# Format and print the average classification report
headers = ["precision", "recall", "f1-score", "support"]
table = []
for label in ['0', '1', 'macro avg', 'weighted avg']:
    if label in avg_report:
        row = [
            label,
            f"{avg_report[label]['precision']:.4f}",
            f"{avg_report[label]['recall']:.4f}",
            f"{avg_report[label]['f1-score']:.4f}",
            f"{int(avg_report[label]['support']):d}"
        ]
        table.append(row)

print(tabulate(table, headers=headers, floatfmt=".4f", numalign="right"))
print(f"\nAverage Accuracy: {avg_report['accuracy']:.4f}")

                precision    recall    f1-score    support
------------  -----------  --------  ----------  ---------
0                  0.9730    0.9369      0.9545      11513
1                  0.7577    0.8826      0.8144       2556
macro avg          0.8653    0.9098      0.8844      14070
weighted avg       0.9339    0.9269      0.9290      14070

Average Accuracy: 0.9269


## Train Final Model on Full Dataset

In [7]:
# Train Final Model on Full Dataset
final_model = xgb.XGBClassifier(
    learning_rate=LR,
    max_depth=MAX_DEPTH,
    n_estimators=int(np.mean(best_iterations)),  # Use average of best iterations from CV
    reg_lambda=L2_REG,  # L2 regularization
    eval_metric='error',  # Classification error rate (1 - accuracy)
    enable_categorical=False,  # Categorical features are already encoded
    scale_pos_weight=scale_pos_weight,  # Handle class imbalance
    random_state=42,
    min_child_weight = MIN_CHILD_WEIGHT, # Minimum sum of instance weight needed in a child
    colsample_bytree = COSAMPLE_RATE, # Subsample ratio of columns when constructing each tree
    subsample = DATA_RATE # Subsample ratio of the training instances
)

final_model.fit(X, y, verbose=False)

,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,0.8
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,'error'


In [8]:
# Predict on test data
test_preds = final_model.predict(test_data)

# Prepare submission dataframe
submission_df = pd.DataFrame({
    "id": test_ids,
    "Depression": test_preds
})

# Save submission file
submission_df.to_csv("../data/HCMUS-22KHDL-Cloud_Djata_xgboost_v2.csv", index=False)